# WGAN-GP NON LINEAR LAYER

This notebook trains a **Wasserstein GAN with Gradient Penalty (WGAN-GP)** on multifractal texture data stored as `.mat` files.  
Each sample contains a 2-channel (512×512) complex-valued field loaded via `scipy.io`. It will be cutted to 128x128

**Pipeline overview:**
1. Load `.mat` files in parallel → HuggingFace `Dataset`
2. Apply an on-the-fly transform to produce 2-channel PyTorch tensors
3. Train a WGAN-GP (Generator + Critic) with Wasserstein loss + **gradient penalty**
4. Save generated textures as `.mat` for OCTAVE analysis

**Key differences vs vanilla WGAN (weight clipping):**
- Critic weights are **not clipped** — instead a **gradient penalty** enforces the Lipschitz constraint
- `λ_GP * E[(||∇C(x̂)||₂ − 1)²]` is added to the critic loss (Gulrajani et al., 2017)
- Optimizer: **Adam** (β₁=0, β₂=0.9) instead of RMSprop
- **No BatchNorm** in the Critic (breaks the gradient penalty constraint)


In [1]:
!which python
!python -V
!which pip
!pip -V
!which jupyter
!jupyter --version
#CUDA_VISIBLE_DEVICES = [1]

/local/janccoce/envs/gans/bin/python
Python 3.8.20
/local/janccoce/envs/gans/bin/pip
pip 25.0.1 from /local/janccoce/envs/gans/lib/python3.8/site-packages/pip (python 3.8)
/local/janccoce/envs/gans/bin/jupyter
Selected Jupyter core packages...
IPython          : 8.12.3
ipykernel        : 6.29.5
ipywidgets       : not installed
jupyter_client   : 8.6.3
jupyter_core     : 5.8.1
jupyter_server   : not installed
jupyterlab       : not installed
nbclient         : not installed
nbconvert        : not installed
nbformat         : not installed
notebook         : not installed
qtconsole        : not installed
traitlets        : 5.14.3


In [2]:
!python -m pip show joblib
!python -c "import sys; print(sys.executable)"
!python -c "import joblib; print(joblib.__version__)"

Name: joblib
Version: 1.4.2
Summary: Lightweight pipelining with Python functions
Home-page: https://joblib.readthedocs.io
Author: 
Author-email: Gael Varoquaux <gael.varoquaux@normalesup.org>
License: BSD 3-Clause
Location: /local/janccoce/envs/gans/lib/python3.8/site-packages
Requires: 
Required-by: 
/local/janccoce/envs/gans/bin/python
1.4.2


In [3]:
import sys
print(sys.executable)

import site
print(site.getsitepackages())

import joblib
print(joblib.__version__)

/local/janccoce/envs/gans/bin/python
['/local/janccoce/envs/gans/lib/python3.8/site-packages']
1.4.2


## 1 · Environment & Imports

In [4]:
import os
import sys
import argparse

import numpy as np
import scipy.io
from PIL import Image
from joblib import Parallel, delayed
from scipy.io import savemat

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision.utils import save_image

from datasets import Dataset

print("Python:", sys.executable)
print("Version:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

/local/janccoce/envs/gans/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Python: /local/janccoce/envs/gans/bin/python
Version: 3.8.20 | packaged by conda-forge | (default, Sep 30 2024, 17:52:49) 
[GCC 13.3.0]
PyTorch: 2.4.1+cu121
CUDA available: True


## 2 · Configuration

In [5]:
# ── Paths ────────────────────────────────────────────────────────────────────
MAT_FOLDER  = "/local/janccoce/WGANProject/dataV6norm/"
OUTPUT_PATH = "/local/janccoce/WGANProject/outputWNLV6-10"                                 
# from google.colab import drive
# drive.mount('/content/drive')

#MAT_FOLDER  = "/local/kabbas/data_janccoce/"
#OUTPUT_PATH = "/local/kabbas/output_janccoce/"

# ── Training hyperparameters ─────────────────────────────────────────────────
BATCH_SIZE      = 32
NOISE_CHANNELS  = 2      # was NOISE_DIM
NOISE_HEIGHT    = 16
NOISE_WIDTH     = 16
NUM_EPOCHS      = 32
LEARNING_RATE   = 1e-4   # Adam lr for WGAN-GP
BETA1           = 0.0    # Adam β₁ for WGAN-GP (paper recommends 0)
BETA2           = 0.9    # Adam β₂ for WGAN-GP
NUM_WORKERS     = 4

# ── WGAN-GP specific ──────────────────────────────────────────────────────────
N_CRITIC        = 5
LAMBDA_GP       = 10     # Gradient penalty coefficient (Gulrajani et al., 2017)

# ── Logging & checkpointing ──────────────────────────────────────────────────
SAMPLE_SIZE     = 2      # number of samples for export (if needed)
LOG_INTERVAL    = 50
SAVE_EPOCH      = 8

# ── Data ─────────────────────────────────────────────────────────────────────
IMAGE_KEY       = "datatmp"
BAND_INDEX      = 0
CHANNELS_IMG    = 2
TARGET_SIZE     = (128, 128)

# Create output directories
for sub in ["", "samples", "checkpoints", "matlab_data"]:
    os.makedirs(os.path.join(OUTPUT_PATH, sub), exist_ok=True)

print("Output directory ready:", OUTPUT_PATH)


Output directory ready: /local/janccoce/WGANProject/outputWNLV6-10


## 3 · Data Loading

`.mat` files are loaded in parallel with `joblib` and assembled into a HuggingFace `Dataset`.  
Each sample exposes two fields:
- `image` — a PIL grayscale preview (band 0, uint8)
- `raw_data` — the full float32 array `(H, W, C)` used for training

In [6]:
from scipy.io import loadmat
fp = loadmat(MAT_FOLDER+'MRW2D_Param1_1004.mat')

print(fp)

{'__header__': b'MATLAB 5.0 MAT-file, written by Octave 9.4.0, 2026-05-19 13:00:21 UTC', '__version__': '1.0', '__globals__': [], 'datatmp': array([[[-0.00738295,  0.15350891],
        [ 0.04473816,  0.10141803],
        [ 0.02596672,  0.09821769],
        ...,
        [ 0.01957108,  0.20868774],
        [-0.03244847,  0.01551875],
        [-0.01471173,  0.16028291]],

       [[-0.1291015 ,  0.14550742],
        [-0.04896135,  0.05189307],
        [-0.04003371, -0.04262506],
        ...,
        [-0.03760931,  0.12060948],
        [-0.07569636,  0.09920703],
        [-0.17215328,  0.06157107]],

       [[-0.05319779, -0.06946612],
        [-0.09874058,  0.09574043],
        [-0.18918471,  0.07626912],
        ...,
        [ 0.07795368,  0.05965788],
        [-0.0649161 ,  0.03366319],
        [-0.03753539,  0.02586771]],

       ...,

       [[-0.06534737,  0.11243486],
        [-0.03075507,  0.1292789 ],
        [-0.0422693 ,  0.13072202],
        ...,
        [ 0.04504361,  0.0091200

In [7]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

#def load_octave_complex_matrix(path, variable_name="datatmp"):

class MatDataset(torch.utils.data.Dataset):
    def __init__(self, mat_folder, image_key="datatmp"):
        self.paths = sorted([
            os.path.join(mat_folder, f)
            for f in os.listdir(mat_folder)
            if f.endswith(".mat")
        ])
        self.image_key = image_key
    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]

        #arr = load_octave_complex_matrix(path, variable_name=self.image_key)  # (H, W, C), complex
        mat_data = loadmat(path)

        # Extract the complex array
        arr = mat_data[self.image_key]

        arr = arr.real.astype(np.float32)            # (H, W, C)

        x = torch.from_numpy(arr).permute(2, 0, 1)      # (C, H, W)
        #x = x[:, :128, :128]                                # (C, 128, 128)
        return x

## Load other dataset to contrast dist(otherDataset,Gen)

In [37]:
class MatDataset2(torch.utils.data.Dataset):
    def __init__(self, mat_folder, image_key="datatmp"):
        self.paths = sorted([
            os.path.join(mat_folder, f)
            for f in os.listdir(mat_folder)
            if f.endswith(".mat")
        ])
        self.image_key = image_key
    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]

        #arr = load_octave_complex_matrix(path, variable_name=self.image_key)  # (H, W, C), complex
        mat_data = loadmat(path)

        # Extract the complex array
        arr = mat_data[self.image_key]

        arr = arr.real.astype(np.float32)            # (H, W, C)

        x = torch.from_numpy(arr).permute(2, 0, 1)      # (C, H, W)
        x = x[:, :128, :128]                                # (C, 128, 128)
        return x
    
MAT_FOLDER2  = "/local/janccoce/DCGANProject/dataV1norm/"
# Load dataset
dataset2 = MatDataset2(
    mat_folder=MAT_FOLDER2,
    image_key=IMAGE_KEY
)

In [38]:
# Load dataset
dataset = MatDataset(
    mat_folder=MAT_FOLDER,
    image_key=IMAGE_KEY
)

# Split into train (80%) and validation (20%) in order
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size

# Create subsets (first 80% for training, last 20% for validation)
train_dataset = torch.utils.data.Subset(dataset, range(train_size))
val_dataset = torch.utils.data.Subset(dataset, range(train_size, len(dataset)))

print(f"Total dataset size: {len(dataset)}")
print(f"Training set size : {len(train_dataset)} (first 80%)")
print(f"Validation set size: {len(val_dataset)} (last 20%)")

sample = dataset[10]

print(f"Sample shape : {sample.shape}")
print(f"Value range  : [{sample.min().item():.3f}, {sample.max().item():.3f}]")

CHANNELS_IMG = sample.shape[0]
print(f"Channels used for training: {CHANNELS_IMG}")

Total dataset size: 10000
Training set size : 8000 (first 80%)
Validation set size: 2000 (last 20%)
Sample shape : torch.Size([2, 128, 128])
Value range  : [-0.505, 0.599]
Channels used for training: 2


In [9]:
path = dataset.paths[0]

mat_data = loadmat(path)
print("Keys in .mat file:", mat_data.keys())
print("Shape of datatmp:", mat_data['datatmp'].shape)
print("Data type:", mat_data['datatmp'].dtype)

Keys in .mat file: dict_keys(['__header__', '__version__', '__globals__', 'datatmp'])
Shape of datatmp: (128, 128, 2)
Data type: float64


## 4 · Model Architecture

### 4.1 · Critic (WGAN-GP Discriminator)

Four strided-convolution blocks (Conv → **no BatchNorm** → LeakyReLU → Dropout) followed by a linear head.  
**BatchNorm is removed** from the Critic — it interferes with the gradient penalty by correlating
samples within a batch and breaking the per-sample Lipschitz constraint.  
**No Sigmoid** — the Critic outputs a raw, unbounded score that estimates the Wasserstein distance.


In [10]:
class Critic(nn.Module):
    """WGAN-GP Critic for (C, 128, 128) inputs.

    BatchNorm is intentionally removed: it breaks the per-sample gradient
    penalty constraint by correlating activations across the batch.
    Outputs unbounded scalar score (no Sigmoid).
    """

    def __init__(self, channels_img=2, img_size=128):
        super().__init__()

        self.conv_blocks = nn.Sequential(
            self._block(channels_img,  32),
            self._block(32,  32),
            self._block(32,  64),
            self._block(64,  64),
        )

        # Infer flattened size dynamically
        with torch.no_grad():
            dummy = torch.zeros(1, channels_img, img_size, img_size)
            flat  = self.conv_blocks(dummy).view(1, -1).shape[1]
        print(f"Critic: flattened size = {flat}")

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat, 1),
            # No Sigmoid — Wasserstein loss requires unbounded output
        )

    @staticmethod
    def _block(in_ch, out_ch, kernel_size=3, stride=2, padding=0):
        """Conv block without BatchNorm (required for WGAN-GP)."""
        return nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size, stride, padding, bias=True),
            nn.LeakyReLU(0.2, inplace=True),
            nn.Dropout(0.25),
        )

    def forward(self, x):
        return self.fc(self.conv_blocks(x))


### 4.2 · Generator

Noise `(N, 2x64x64)` → Conv + BN + ReLU → reshape `(N, 512, 16, 16)` → three Upsample-Conv blocks → `(N, C, 512, 512)` with Tanh.  

In [11]:
class Generator(nn.Module):
    """WGAN-NoLinV2: noise 2x64x64 → texture 2x512x512."""
    def __init__(self, in_channels=2, out_channels=2):
        super().__init__()

        self.initial = nn.Sequential(
            nn.Conv2d(in_channels, 512, kernel_size=15, padding=0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
        )

        # Upsampling blocks
        self.up1 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),   # 64 → 128
            nn.Conv2d(512, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )

        self.up2 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),   # 128 → 256
            nn.Conv2d(64, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )

        self.up3 = nn.Sequential(
            nn.Upsample(scale_factor=2, mode='nearest'),   # 256 → 512
            nn.Conv2d(32, out_channels, kernel_size=3, padding=1),
            nn.Tanh(),  # output in [-1, 1]
        )

    def forward(self, x):
        # x shape: (N, 2, 64, 64)
        x = self.initial(x)   # (N, 512, 64, 64)
        x = self.up1(x)       # (N, 64, 128, 128)
        x = self.up2(x)       # (N, 32, 256, 256)
        x = self.up3(x)       # (N, out_channels, 512, 512)
        return x

### 4.3 · Weight Initialisation


Xavier Uniform takes uniform[sqrt(6/...), ...]

In [12]:
def initialize_weights(model):
    """Applies Xavier uniform init to Conv/Linear layers and DCGAN init to BatchNorm."""
    for m in model.modules():
        if isinstance(m, (nn.ConvTranspose2d, nn.Conv2d, nn.Linear)):
            nn.init.xavier_uniform_(m.weight) #uniform[sqrt(6/...), ...]
            if m.bias is not None:
                nn.init.constant_(m.bias, 0) #As in the article of reference
        elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
            nn.init.normal_(m.weight, 1.0, 0.02)
            nn.init.constant_(m.bias, 0)

### 4.4 · Model Instantiation

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ngpu = torch.cuda.device_count() if device.type == "cuda" else 1
print(f"Device: {device}  |  GPUs: {ngpu}")

netG = Generator(in_channels=NOISE_CHANNELS, out_channels=CHANNELS_IMG).to(device)
netC = Critic(channels_img=CHANNELS_IMG).to(device)  # Critic expects channels_img

if device.type == "cuda" and ngpu > 1:
    netG = nn.DataParallel(netG, list(range(ngpu)))
    netC = nn.DataParallel(netC, list(range(ngpu)))

netG.apply(initialize_weights)
netC.apply(initialize_weights)

print("\n── Generator architecture ──────────────────")
print(netG)
print("\n── Critic architecture ─────────────────────")
print(netC)

Device: cuda  |  GPUs: 1
Critic: flattened size = 3136

── Generator architecture ──────────────────
Generator(
  (initial): Sequential(
    (0): Conv2d(2, 512, kernel_size=(15, 15), stride=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
  )
  (up1): Sequential(
    (0): Upsample(scale_factor=2.0, mode='nearest')
    (1): Conv2d(512, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU(inplace=True)
  )
  (up2): Sequential(
    (0): Upsample(scale_factor=2.0, mode='nearest')
    (1): Conv2d(64, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): ReLU(inplace=True)
  )
  (up3): Sequential(
    (0): Upsample(scale_factor=2.0, mode='nearest')
    (1): Conv2d(32, 2, kernel_si

## 5 · Training

**WGAN-GP** training loop (Gulrajani et al., 2017 — *Improved Training of Wasserstein GANs*):
- **Critic step** (repeated `n_critic` times):
  1. Compute `lossC = E[C(fake)] − E[C(real)] + λ_GP · E[(||∇C(x̂)||₂ − 1)²]`
  2. `x̂ = ε·real + (1−ε)·fake` for random `ε ~ Uniform[0,1]` (interpolated samples)
  3. **No weight clipping** — the gradient penalty alone enforces the Lipschitz constraint
- **Generator step**: minimise `−E[C(G(z))]`

Optimiser: **Adam** (β₁=0, β₂=0.9, lr=1e-4) — recommended in the WGAN-GP paper.  
Gradient penalty coefficient: **λ_GP = 10** (default from the paper).


In [14]:
def get_latest_epoch(checkpoint_dir):
    """Return the highest epoch number found in *checkpoint_dir*, or -1."""
    ckpts = [
        f for f in os.listdir(checkpoint_dir)
        if f.startswith("generator_epoch_") and f.endswith(".pth")
    ]
    if not ckpts:
        return -1
    epochs = [int(f.split("_")[-1].split(".")[0]) for f in ckpts]
    return max(epochs)

In [15]:
def save_octave_matrix(arr, path, variable_name="datatmp"):
    """
    Saves a 3D NumPy array (H, W, C) to an Octave-compatible .mat text file.
    Assumes real-valued data and saves it in complex format (real_part, 0.0).
    """
    if arr.ndim != 3:
        raise ValueError(f"Input array must be 3D (H, W, C), but got {arr.ndim}D.")

    H, W, C = arr.shape

    with open(path, "w") as f:
        f.write(f"# Created by Python script\n")
        f.write(f"# name: {variable_name}\n")
        f.write(f"# type: complex matrix\n")
        f.write(f"# ndims: 3\n")
        f.write(f" {H} {W} {C}\n")

        # Iterate through channels, then height, then width to match Octave's column-major order implicitly
        # Octave stores complex matrices as (real, imag) pairs
        for k in range(C):
            for j in range(W):
                for i in range(H):
                    val = arr[i, j, k]
                    f.write(f"({val},0.0)\n")


def generate_and_export(
    output_dir,
    noise_channels,
    noise_height,
    noise_width,
    channels_img,
    sample_size=20,
    epoch=None,
    gen_model=None,
    output_resolution=None,          # optional tag, e.g., 512 or 1024
):
    """
    Export synthetic textures to Octave .mat files.
    
    The generator's output size = noise_size * 8 because there are 3 upsampling stages (×2 each).
    Example: noise (64x64) -> output (512x512), noise (128x128) -> output (1024x1024).
    
    Args:
        output_resolution: If provided, used in folder naming (e.g., "epoch_003_1024").
                           If None, auto-detected as noise_height*8.
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    if gen_model is None:
        checkpoint_dir = os.path.join(output_dir, "checkpoints")
        if not os.path.isdir(checkpoint_dir):
            print(f"[ERROR] Checkpoint directory not found: {checkpoint_dir}")
            return
        if epoch is None:
            epoch = get_latest_epoch(checkpoint_dir)
            if epoch < 0:
                print("[ERROR] No generator checkpoints found to load.")
                return
        ckpt_path = os.path.join(checkpoint_dir, f"generator_epoch_{epoch}.pth")
        if not os.path.isfile(ckpt_path):
            print(f"[ERROR] Checkpoint not found: {ckpt_path}")
            return
        print(f"Loading checkpoint: {ckpt_path}")
        # Note: Generator expects in_channels, out_channels
        gen = Generator(in_channels=noise_channels, out_channels=channels_img).to(device)
        gen.load_state_dict(torch.load(ckpt_path, map_location=device))
    else:
        gen = gen_model

    # Determine output spatial size
    out_height = noise_height * 8
    out_width  = noise_width * 8
    if output_resolution is None:
        output_resolution = f"{out_height}x{out_width}"
    else:
        output_resolution = str(output_resolution)

    gen.eval()
    with torch.no_grad():
        # Spatial noise
        noise = torch.randn(sample_size, noise_channels, noise_height, noise_width, device=device)
        fake_samples = gen(noise).cpu().numpy()   # shape: (sample_size, C, out_height, out_width)

    print(f"Generated tensor shape: {fake_samples.shape} (output resolution {output_resolution})")

    # Build output folder including resolution tag
    if epoch is not None:
        generation_sub_dir = f"epoch_{epoch:03d}_{output_resolution}"
    else:
        generation_sub_dir = f"final_generation_{output_resolution}"

    out_dir = os.path.join(output_dir, "matlab_data", generation_sub_dir)
    os.makedirs(out_dir, exist_ok=True)

    for i in range(sample_size):
        single_sample = fake_samples[i]                     # (C, H, W)
        single_sample_hwc = np.transpose(single_sample, (1, 2, 0))  # (H, W, C)
        filename = f"MRW2DGEN{1001 + i:04d}.mat"
        file_path = os.path.join(out_dir, filename)
        save_octave_matrix(single_sample_hwc, file_path, variable_name="datatmp")
    print(f"Saved Octave .mat  → {file_path}")

In [16]:
def compute_gradient_penalty(critic, real_imgs, fake_imgs, device):
    """Gradient penalty for WGAN-GP (Gulrajani et al., 2017).

    Enforces the 1-Lipschitz constraint on the critic by penalising
    gradients that deviate from unit norm, evaluated on interpolated
    samples between real and generated distributions.

    Args:
        critic     : the Critic network
        real_imgs  : batch of real images,  shape (N, C, H, W)
        fake_imgs  : batch of fake images,  shape (N, C, H, W)
        device     : torch device

    Returns:
        Scalar gradient penalty (before multiplying by λ_GP).
    """
    batch_size = real_imgs.size(0)

    # Random interpolation coefficient ε ~ Uniform[0, 1]
    alpha = torch.rand(batch_size, 1, 1, 1, device=device)

    # Interpolated samples x̂ = ε·real + (1−ε)·fake
    interpolated = (alpha * real_imgs + (1.0 - alpha) * fake_imgs).requires_grad_(True)

    # Critic score on interpolated samples
    d_interp = critic(interpolated)

    # Gradients of critic output w.r.t. interpolated inputs
    gradients = torch.autograd.grad(
        outputs=d_interp,
        inputs=interpolated,
        grad_outputs=torch.ones_like(d_interp),
        create_graph=True,
        retain_graph=True,
        only_inputs=True,
    )[0]  # shape: (N, C, H, W)

    # Flatten to (N, C*H*W) and compute L2 norm per sample
    gradients = gradients.view(batch_size, -1)
    gradient_penalty = (torch.relu(gradients.norm(2, dim=1) - 1.0) ** 2).mean()
    return gradient_penalty


In [17]:
def train(args):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device        : {device}")
    print(f"Dataset size  : {len(train_dataset)}")
    print(f"Channels      : {args.channels_img}")
    pad = 14 #Doing noise + pad in order to avoid padding with a lot of zeros
    dataloader = DataLoader(
        train_dataset,
        batch_size=args.batch_size,
        shuffle=True,
        num_workers=args.num_workers
    )

    batch_check = next(iter(dataloader))
    print(f"Batch shape   : {batch_check.shape}")

    # --- Models ---------------------------------------------------------
    netC_ = Critic(channels_img=args.channels_img).to(device)
    # Generator now uses spatial noise dimensions
    netG_ = Generator(
        in_channels=args.noise_channels,
        out_channels=args.channels_img
    ).to(device)

    netG_.apply(initialize_weights)
    netC_.apply(initialize_weights)

    # --- WGAN-GP optimizers (Adam, β₁=0, β₂=0.9 as per paper) ----------
    optimizerC = optim.Adam(netC_.parameters(), lr=args.lr, betas=(0.0, 0.9))
    optimizerG = optim.Adam(netG_.parameters(), lr=args.lr, betas=(0.0, 0.9))

    start_epoch = 0
    if args.resume_training:
        checkpoint_dir = os.path.join(args.output_dir, "checkpoints")
        latest_epoch = get_latest_epoch(checkpoint_dir)
        if latest_epoch >= 0:
            print(f"Resuming training from epoch {latest_epoch}")
            netG_.load_state_dict(torch.load(os.path.join(checkpoint_dir, f"generator_epoch_{latest_epoch}.pth"), map_location=device))
            netC_.load_state_dict(torch.load(os.path.join(checkpoint_dir, f"critic_epoch_{latest_epoch}.pth"), map_location=device))
            start_epoch = latest_epoch + 1
        else:
            print("No checkpoints found. Starting new training.")

    # Fixed noise
    fixed_noise_128 = torch.randn(
        1,
        args.noise_channels,
        args.noise_height+pad,   # 7 + 16 + 7 = 30
        args.noise_width+pad,    # 7 + 16 + 7 = 30
        device=device
    )
    
    fixed_noise_256 = torch.randn(
        1,
        args.noise_channels,
        args.noise_height*2+pad,
        args.noise_width*2+pad,
        device=device
    )

    fixed_noise_512 = torch.randn(
        1,
        args.noise_channels,
        args.noise_height*4+pad,
        args.noise_width*4+pad,
        device=device
    )
    
    fixed_noise_1024 = torch.randn(
        1,
        args.noise_channels,
        args.noise_height*8+pad,
        args.noise_width*8+pad,
        device=device
    )

    print("\nStarting WGAN-GP training...")
    for epoch in range(start_epoch, args.num_epochs):
        for i, real_imgs in enumerate(dataloader):
            real_imgs = real_imgs.to(device)
            batch_size = real_imgs.size(0)

            # --- Critic steps --------------------------------------------
            for _ in range(args.n_critic):
                # Spatial noise: (batch, noise_channels, H, W)
                noise = torch.randn(
                    batch_size,
                    args.noise_channels,
                    args.noise_height+pad,
                    args.noise_width+pad,
                    device=device
                )
                fake_imgs = netG_(noise).detach()

                netC_.zero_grad()

                # Wasserstein loss terms
                loss_real = netC_(real_imgs).mean()
                loss_fake = netC_(fake_imgs).mean()

                # Gradient penalty (enforces 1-Lipschitz constraint)
                gp = compute_gradient_penalty(netC_, real_imgs, fake_imgs, device)

                # Total critic loss: minimise E[C(fake)] − E[C(real)] + λ_GP * GP
                lossC = loss_fake - loss_real + args.lambda_gp * gp
                lossC.backward()
                optimizerC.step()

            # --- Generator step ------------------------------------------
            netG_.zero_grad()
            noise = torch.randn(
                batch_size,
                args.noise_channels,
                args.noise_height+pad,
                args.noise_width+pad,
                device=device
            )
            fake_imgs = netG_(noise)
            lossG = -netC_(fake_imgs).mean()
            lossG.backward()
            optimizerG.step()

            if i % args.log_interval == 0:
                print(
                    f"[Epoch {epoch:03d}/{args.num_epochs}] "
                    f"[Batch {i:03d}/{len(dataloader)}] "
                    f"Loss_C: {lossC.item():.4f}  Loss_G: {lossG.item():.4f}  "
                    f"W_dist: {-lossC.item():.4f}"
                )

        # ── End‑of‑epoch: save samples ──
        with torch.no_grad():
            
            # --- 128x128 generation ---
            fake_128 = netG_(fixed_noise_128).detach().cpu()
            fake_128 = (fake_128 + 1) / 2
            print(f"Epoch {epoch:03d} – 128x128 output shape: {fake_128.shape}")
            ch0_128 = fake_128[0:1, 0:1, :, :]
            ch1_128 = fake_128[0:1, 1:2, :, :]
            side_128 = torch.cat([ch0_128, ch1_128], dim=3)
            save_image(
                side_128,
                os.path.join(args.output_dir, "samples", f"epoch_{epoch:03d}_128_both_channels.png"),
                normalize=False,
            )
            
            # --- 256x256 generation ---
            fake_256 = netG_(fixed_noise_256).detach().cpu()
            fake_256 = (fake_256 + 1) / 2
            print(f"Epoch {epoch:03d} – 256x256 output shape: {fake_256.shape}")
            ch0_256 = fake_256[0:1, 0:1, :, :]
            ch1_256 = fake_256[0:1, 1:2, :, :]
            side_256 = torch.cat([ch0_256, ch1_256], dim=3)
            save_image(
                side_256,
                os.path.join(args.output_dir, "samples", f"epoch_{epoch:03d}_256_both_channels.png"),
                normalize=False,
            )
            
            # --- 512x512 generation ---
            fake_512 = netG_(fixed_noise_512).detach().cpu()
            fake_512 = (fake_512 + 1) / 2
            ch0_512 = fake_512[0:1, 0:1, :, :]
            ch1_512 = fake_512[0:1, 1:2, :, :]
            side_512 = torch.cat([ch0_512, ch1_512], dim=3)
            save_image(
                side_512,
                os.path.join(args.output_dir, "samples", f"epoch_{epoch:03d}_512_both_channels.png"),
                normalize=False,
            )

            # --- 1024x1024 generation ---
            fake_1024 = netG_(fixed_noise_1024).detach().cpu()
            fake_1024 = (fake_1024 + 1) / 2
            print(f"Epoch {epoch:03d} – 1024x1024 output shape: {fake_1024.shape}")
            ch0_1024 = fake_1024[0:1, 0:1, :, :]
            ch1_1024 = fake_1024[0:1, 1:2, :, :]
            side_1024 = torch.cat([ch0_1024, ch1_1024], dim=3)
            save_image(
                side_1024,
                os.path.join(args.output_dir, "samples", f"epoch_{epoch:03d}_1024_both_channels.png"),
                normalize=False,
            )

        # --- Checkpointing ----------------------------------------------
        if epoch % args.save_epoch == 0 or epoch == args.num_epochs - 1:
            ckpt = os.path.join(args.output_dir, "checkpoints")
            torch.save(netG_.state_dict(), os.path.join(ckpt, f"generator_epoch_{epoch}.pth"))
            torch.save(netC_.state_dict(), os.path.join(ckpt, f"critic_epoch_{epoch}.pth"))

        # --- Generate and export
        print(f"Generating and exporting Octave documents for epoch {epoch}...")
        generate_and_export(
            output_dir     = args.output_dir,
            noise_channels = args.noise_channels,
            noise_height   = args.noise_height+pad,
            noise_width    = args.noise_height+pad,
            channels_img   = 2,
            sample_size    = args.sample_size*4,
            epoch          = epoch,
            gen_model      = netG_,
            output_resolution = "128"
        )
        
        generate_and_export(
            output_dir     = args.output_dir,
            noise_channels = args.noise_channels,
            noise_height   = args.noise_height*2+pad,
            noise_width    = args.noise_height*2+pad,
            channels_img   = 2,
            sample_size    = args.sample_size*3,
            epoch          = epoch,
            gen_model      = netG_,
            output_resolution = "256"
        )
        generate_and_export(
            output_dir     = args.output_dir,
            noise_channels = args.noise_channels,
            noise_height   = args.noise_height*4+pad,
            noise_width    = args.noise_height*4+pad,
            channels_img   = 2,
            sample_size    = args.sample_size*2,
            epoch          = epoch,
            gen_model      = netG_,
            output_resolution = "512"
        )
        generate_and_export(
            output_dir     = args.output_dir,
            noise_channels = args.noise_channels,
            noise_height   = args.noise_height*8+pad,
            noise_width    = args.noise_height*8+pad,
            channels_img   = 2,
            sample_size    = args.sample_size,
            epoch          = epoch,
            gen_model      = netG_,
            output_resolution = "1024"
        )

    print("Training finished.")
    return netG_, netC_


# The Training

In [18]:
MAT_FOLDER = "/local/janccoce/WGANProject/dataV6norm/"
OUTPUT_PATH = f"/local/janccoce/WGANProject/outputWNLV6-10"

for sub in ["", "samples", "checkpoints", "matlab_data"]:
        os.makedirs(os.path.join(OUTPUT_PATH, sub), exist_ok=True)
print("Output directory ready:", OUTPUT_PATH)
    
# Clean up any stale model objects before launching training
for _name in ["netC", "netG"]:
    if _name in dir():
        del globals()[_name]

args = argparse.Namespace(
    data_root       = MAT_FOLDER,
    output_dir      = OUTPUT_PATH,
    batch_size      = 32,
    noise_channels  = 2,
    noise_height    = 16, #The +14 is added inside the previous function, so we don't need to add 30 here
    noise_width     = 16,
    num_epochs      = 32,
    lr              = LEARNING_RATE,
    beta1           = BETA1,
    num_workers     = 6,
    sample_size     = 5,
    log_interval    = 100,
    save_epoch      = 2,
    channels_img    = CHANNELS_IMG,
    n_critic        = 5,
    lambda_gp       = LAMBDA_GP,   # gradient penalty coefficient
    resume_training = True
)

netG, netC = train(args)


Output directory ready: /local/janccoce/WGANProject/outputWNLV6-10
Device        : cuda
Dataset size  : 8000
Channels      : 2
Batch shape   : torch.Size([32, 2, 128, 128])


KeyboardInterrupt: 